In [ ]:
import sys
import importlib
from pathlib import Path
from datetime import datetime
import json
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

target_path = "/content/drive/MyDrive/Thesis/models"

if target_path not in sys.path:
    sys.path.insert(0, target_path)

importlib.invalidate_caches()

from utils import DATA_DIR, EKMAN_LABELS, set_all_seeds
from evaluate import compute_metrics

from train_phase3 import (
    GoEmotionsDataset,
    hf_compute_metrics,
    load_data,
    MODEL_NAME,
    MAX_LENGTH,
    BATCH_SIZE,
    N_EPOCHS,
    LEARNING_RATE,
    WEIGHT_DECAY,
    WARMUP_STEPS,
    NUM_LABELS,
)

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SEEDS = [42, 0, 1]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
def make_json_safe(value):
    if isinstance(value, dict):
        return {
            str(key): make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    return value


def get_class_counts(labels, num_labels=NUM_LABELS):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    counts = np.bincount(
        labels,
        minlength=num_labels,
    )

    if np.any(counts == 0):
        missing = np.where(counts == 0)[0].tolist()

        raise ValueError(
            f"Missing classes in training labels: {missing}"
        )

    return counts


def normalize_by_expected_example_weight(
    weights,
    counts,
):
    weights = np.asarray(
        weights,
        dtype=np.float64,
    )

    counts = np.asarray(
        counts,
        dtype=np.float64,
    )

    expected_weight = (
        np.sum(weights * counts)
        / np.sum(counts)
    )

    return weights / expected_weight


def save_confusion_matrix_to_folder(
    y_true,
    y_pred,
    labels,
    output_path,
    title,
):
    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(labels))),
    )

    figure, axis = plt.subplots(
        figsize=(9, 8)
    )

    image = axis.imshow(
        matrix,
        interpolation="nearest",
        cmap="Blues",
    )

    figure.colorbar(
        image,
        ax=axis,
    )

    axis.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        xlabel="Predicted label",
        ylabel="True label",
        title=title,
    )

    plt.setp(
        axis.get_xticklabels(),
        rotation=45,
        ha="right",
        rotation_mode="anchor",
    )

    threshold = matrix.max() / 2 if matrix.size else 0

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            axis.text(
                column_index,
                row_index,
                format(matrix[row_index, column_index], "d"),
                ha="center",
                va="center",
                color=(
                    "white"
                    if matrix[row_index, column_index] > threshold
                    else "black"
                ),
            )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(figure)

    return matrix

In [ ]:
def balanced_inverse_frequency_weights(
    labels,
    num_labels=NUM_LABELS,
):
    counts = get_class_counts(
        labels,
        num_labels=num_labels,
    )

    n_examples = counts.sum()
    n_classes = num_labels

    weights = n_examples / (
        n_classes * counts
    )

    weights = normalize_by_expected_example_weight(
        weights,
        counts,
    )

    return weights, counts


def sqrt_inverse_frequency_weights(
    labels,
    num_labels=NUM_LABELS,
):
    balanced_weights, counts = balanced_inverse_frequency_weights(
        labels,
        num_labels=num_labels,
    )

    weights = np.sqrt(
        balanced_weights
    )

    weights = normalize_by_expected_example_weight(
        weights,
        counts,
    )

    return weights, counts


def capped_inverse_frequency_weights(
    labels,
    cap=5.0,
    num_labels=NUM_LABELS,
):
    balanced_weights, counts = balanced_inverse_frequency_weights(
        labels,
        num_labels=num_labels,
    )

    weights = np.minimum(
        balanced_weights,
        cap,
    )

    weights = normalize_by_expected_example_weight(
        weights,
        counts,
    )

    return weights, counts


def effective_number_weights(
    labels,
    beta=0.999,
    num_labels=NUM_LABELS,
):
    counts = get_class_counts(
        labels,
        num_labels=num_labels,
    ).astype(np.float64)

    effective_num = (
        1.0 - np.power(beta, counts)
    ) / (1.0 - beta)

    weights = 1.0 / effective_num

    weights = normalize_by_expected_example_weight(
        weights,
        counts,
    )

    return weights, counts


def power_capped_weights(
    labels,
    power=0.5,
    cap=5.0,
    num_labels=NUM_LABELS,
):
    balanced_weights, counts = balanced_inverse_frequency_weights(
        labels,
        num_labels=num_labels,
    )

    weights = np.power(
        balanced_weights,
        power,
    )

    weights = np.minimum(
        weights,
        cap,
    )

    weights = normalize_by_expected_example_weight(
        weights,
        counts,
    )

    return weights, counts


def compute_weight_scheme(
    labels,
    scheme_name,
    **kwargs,
):
    if scheme_name == "sqrt_inverse_frequency":
        return sqrt_inverse_frequency_weights(
            labels,
        )

    if scheme_name == "capped_inverse_frequency":
        return capped_inverse_frequency_weights(
            labels,
            cap=kwargs.get("cap", 5.0),
        )

    if scheme_name == "effective_number":
        return effective_number_weights(
            labels,
            beta=kwargs.get("beta", 0.999),
        )

    if scheme_name == "power_capped":
        return power_capped_weights(
            labels,
            power=kwargs.get("power", 0.5),
            cap=kwargs.get("cap", 5.0),
        )

    raise ValueError(
        f"Unknown weighting scheme: {scheme_name}"
    )

In [ ]:
class WeightedCETrainer(Trainer):
    def __init__(
        self,
        *args,
        class_weights=None,
        **kwargs,
    ):
        if class_weights is None:
            raise ValueError(
                "class_weights must be provided."
            )

        super().__init__(
            *args,
            **kwargs,
        )

        if not isinstance(
            class_weights,
            torch.Tensor,
        ):
            class_weights = torch.tensor(
                class_weights,
                dtype=torch.float32,
            )

        if class_weights.ndim != 1:
            raise ValueError(
                "class_weights must be one-dimensional."
            )

        if len(class_weights) != NUM_LABELS:
            raise ValueError(
                f"Expected {NUM_LABELS} class weights, "
                f"but got {len(class_weights)}."
            )

        self.class_weights = (
            class_weights
            .detach()
            .clone()
            .to(dtype=torch.float32)
        )

        self.model_accepts_loss_kwargs = False

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs,
    ):
        model_inputs = dict(inputs)

        labels = model_inputs.pop(
            "labels"
        ).long()

        outputs = model(
            **model_inputs
        )

        logits = outputs.logits

        weights = self.class_weights.to(
            device=logits.device,
        )

        loss = F.cross_entropy(
            input=logits,
            target=labels,
            weight=weights,
            reduction="mean",
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )

In [ ]:
def run_class_weight_experiment(
    config,
    seed=42,
    output_root=OUTPUT_ROOT,
):
    set_all_seeds(seed)

    scheme_name = config["scheme_name"]
    experiment_name = config["experiment_name"]

    timestamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )

    run_name = (
        f"phase4_cw_{experiment_name}_"
        f"seed{seed}_{timestamp}"
    )

    out_dir = Path(output_root) / run_name

    if out_dir.exists():
        raise FileExistsError(
            f"Output folder already exists:\n{out_dir}"
        )

    out_dir.mkdir(
        parents=True,
        exist_ok=False,
    )

    print("=" * 80)
    print("PHASE 4 CLASS-WEIGHT CONTROL")
    print("=" * 80)
    print(f"Experiment: {experiment_name}")
    print(f"Scheme:     {scheme_name}")
    print(f"Seed:       {seed}")
    print(f"Output:     {out_dir}")
    print("=" * 80)

    train_df, val_df, test_df = load_data()

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label={
            index: label
            for index, label in enumerate(EKMAN_LABELS)
        },
        label2id={
            label: index
            for index, label in enumerate(EKMAN_LABELS)
        },
    )

    weights, counts = compute_weight_scheme(
        train_df["ekman_id"].values,
        scheme_name=scheme_name,
        **config.get("parameters", {}),
    )

    weights_tensor = torch.tensor(
        weights,
        dtype=torch.float32,
    )

    weight_report = {
        "experiment_name": experiment_name,
        "scheme_name": scheme_name,
        "parameters": config.get("parameters", {}),
        "class_counts": {
            EKMAN_LABELS[index]: int(count)
            for index, count in enumerate(counts)
        },
        "class_weights": {
            EKMAN_LABELS[index]: float(weight)
            for index, weight in enumerate(weights)
        },
        "max_min_ratio": float(
            np.max(weights) / np.min(weights)
        ),
    }

    print("\nClass weights:")
    for index, label in enumerate(EKMAN_LABELS):
        print(
            f"{label:<10} count={int(counts[index]):<6} "
            f"weight={weights[index]:.6f}"
        )

    print(
        f"\nMax/min weight ratio: "
        f"{weight_report['max_min_ratio']:.3f}"
    )

    with open(
        out_dir / "weight_report.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(weight_report),
            file,
            indent=2,
            ensure_ascii=False,
        )

    train_ds = GoEmotionsDataset(
        train_df["text_clean"],
        train_df["ekman_id"],
        tokenizer,
    )

    val_ds = GoEmotionsDataset(
        val_df["text_clean"],
        val_df["ekman_id"],
        tokenizer,
    )

    test_ds = GoEmotionsDataset(
        test_df["text_clean"],
        test_df["ekman_id"],
        tokenizer,
    )

    training_args = TrainingArguments(
        output_dir=str(out_dir / "checkpoints"),
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=N_EPOCHS,
        warmup_steps=WARMUP_STEPS,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        logging_steps=100,
        seed=seed,
        data_seed=seed,
        report_to="none",
    )

    trainer = WeightedCETrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(
            tokenizer=tokenizer
        ),
        compute_metrics=hf_compute_metrics,
        class_weights=weights_tensor,
    )

    print("\nStarting training...")

    training_start = time.time()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    trainer.train()

    training_time = time.time() - training_start

    peak_gpu_memory_gb = None

    if torch.cuda.is_available():
        peak_gpu_memory_gb = (
            torch.cuda.max_memory_allocated() / 1e9
        )

    best_model_dir = out_dir / "best_model"

    trainer.save_model(
        str(best_model_dir)
    )

    tokenizer.save_pretrained(
        str(best_model_dir)
    )

    def evaluate_and_save(
        dataset,
        dataframe,
        split_name,
    ):
        prediction_output = trainer.predict(
            dataset,
            metric_key_prefix=split_name,
        )

        logits = prediction_output.predictions

        if isinstance(logits, tuple):
            logits = logits[0]

        y_true = prediction_output.label_ids.astype(int)
        y_pred = np.argmax(
            logits,
            axis=-1,
        ).astype(int)

        stable_logits = (
            logits
            - np.max(
                logits,
                axis=1,
                keepdims=True,
            )
        )

        exponentials = np.exp(stable_logits)

        probabilities = (
            exponentials
            / np.sum(
                exponentials,
                axis=1,
                keepdims=True,
            )
        )

        split_metrics = compute_metrics(
            y_true,
            y_pred,
            EKMAN_LABELS,
        )

        prediction_columns = {
            "text_clean": dataframe["text_clean"]
                .astype(str)
                .tolist(),
            "true_id": y_true,
            "true_label": [
                EKMAN_LABELS[value]
                for value in y_true
            ],
            "predicted_id": y_pred,
            "predicted_label": [
                EKMAN_LABELS[value]
                for value in y_pred
            ],
            "correct": y_true == y_pred,
        }

        if "id" in dataframe.columns:
            prediction_columns = {
                "id": dataframe["id"].tolist(),
                **prediction_columns,
            }

        predictions_dataframe = pd.DataFrame(
            prediction_columns
        )

        for class_index, class_name in enumerate(EKMAN_LABELS):
            safe_class_name = (
                str(class_name)
                .lower()
                .replace(" ", "_")
            )

            predictions_dataframe[
                f"probability_{safe_class_name}"
            ] = probabilities[:, class_index]

        predictions_dataframe.to_csv(
            out_dir / f"{split_name}_predictions.csv",
            index=False,
        )

        matrix = save_confusion_matrix_to_folder(
            y_true=y_true,
            y_pred=y_pred,
            labels=EKMAN_LABELS,
            output_path=(
                out_dir
                / f"{split_name}_confusion_matrix.png"
            ),
            title=f"{split_name.capitalize()} confusion matrix",
        )

        np.savetxt(
            out_dir / f"{split_name}_confusion_matrix.csv",
            matrix,
            delimiter=",",
            fmt="%d",
        )

        with open(
            out_dir / f"{split_name}_metrics.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                make_json_safe(split_metrics),
                file,
                indent=2,
                ensure_ascii=False,
            )

        return split_metrics

    validation_metrics = evaluate_and_save(
        val_ds,
        val_df,
        "validation",
    )

    test_metrics = evaluate_and_save(
        test_ds,
        test_df,
        "test",
    )

    complete_results = {
        "run_name": run_name,
        "experiment_name": experiment_name,
        "treatment": "class_weighted_cross_entropy_control",
        "seed": seed,
        "model_name": MODEL_NAME,
        "weight_report": weight_report,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_validation_macro_f1_during_training": trainer.state.best_metric,
        "training_time_seconds": training_time,
        "peak_gpu_memory_gb": peak_gpu_memory_gb,
        "validation": validation_metrics,
        "test": test_metrics,
    }

    with open(
        out_dir / "complete_results.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(complete_results),
            file,
            indent=2,
            ensure_ascii=False,
        )

    with open(
        out_dir / "training_log_history.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            make_json_safe(trainer.state.log_history),
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("\nValidation macro-F1:")
    print(validation_metrics["macro_f1"])

    print("\nTest macro-F1:")
    print(test_metrics["macro_f1"])

    print("\nResults saved in:")
    print(out_dir)

    return complete_results

In [ ]:
CLASS_WEIGHT_CONFIGS = [
    {
        "experiment_name": "sqrt_inverse_frequency",
        "scheme_name": "sqrt_inverse_frequency",
        "parameters": {},
    },
    {
        "experiment_name": "capped_inverse_frequency_cap3",
        "scheme_name": "capped_inverse_frequency",
        "parameters": {
            "cap": 3.0,
        },
    },
    {
        "experiment_name": "capped_inverse_frequency_cap5",
        "scheme_name": "capped_inverse_frequency",
        "parameters": {
            "cap": 5.0,
        },
    },
    {
        "experiment_name": "effective_number_beta0999",
        "scheme_name": "effective_number",
        "parameters": {
            "beta": 0.999,
        },
    },
    {
        "experiment_name": "effective_number_beta09999",
        "scheme_name": "effective_number",
        "parameters": {
            "beta": 0.9999,
        },
    },

    {
        "experiment_name": "calibrated_power025_cap3",
        "scheme_name": "power_capped",
        "parameters": {
            "power": 0.25,
            "cap": 3.0,
        },
    },
    {
        "experiment_name": "calibrated_power050_cap3",
        "scheme_name": "power_capped",
        "parameters": {
            "power": 0.50,
            "cap": 3.0,
        },
    },
    {
        "experiment_name": "calibrated_power075_cap3",
        "scheme_name": "power_capped",
        "parameters": {
            "power": 0.75,
            "cap": 3.0,
        },
    },
]

In [ ]:
def run_one_class_weight_config_for_all_seeds(
    experiment_name,
    seeds=SEEDS,
):
    matching_configs = [
        config
        for config in CLASS_WEIGHT_CONFIGS
        if config["experiment_name"] == experiment_name
    ]

    if len(matching_configs) != 1:
        raise ValueError(
            f"Could not find exactly one config for: {experiment_name}"
        )

    config = matching_configs[0]

    results = []

    for seed in seeds:
        result = run_class_weight_experiment(
            config=config,
            seed=seed,
            output_root=OUTPUT_ROOT,
        )

        results.append(result)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results

In [ ]:
sqrt_results = run_one_class_weight_config_for_all_seeds(
    "sqrt_inverse_frequency"
)

PHASE 4 CLASS-WEIGHT CONTROL
Experiment: sqrt_inverse_frequency
Scheme:     sqrt_inverse_frequency
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_sqrt_inverse_frequency_seed42_20260702_082553


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6246555295947738

Test macro-F1:
0.6266345630302025

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_sqrt_inverse_frequency_seed1_20260702_084051


In [ ]:
cap3_results = run_one_class_weight_config_for_all_seeds(
    "capped_inverse_frequency_cap3"
)

PHASE 4 CLASS-WEIGHT CONTROL
Experiment: capped_inverse_frequency_cap3
Scheme:     capped_inverse_frequency
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_capped_inverse_frequency_cap3_seed42_20260702_084813


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Class weights:
anger      count=3878   weight=1.675972
disgust    count=497    weight=3.759787
fear       count=515    weight=3.759787
joy        count=12920  weight=0.503051
sadness    count=2121   weight=3.064318
surprise   count=3553   weight=1.829276
neutral    count=12818  weight=0.507054

Max/min weight ratio: 7.474

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.010923,0.956779,0.642622,0.588773,0.653540
2,0.868627,0.936667,0.665054,0.608862,0.672263
3,0.695959,0.966819,0.662415,0.604173,0.667570


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6098321287422882

Test macro-F1:
0.6072085833704869

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_capped_inverse_frequency_cap3_seed1_20260702_090254


In [ ]:
cap5_results = run_one_class_weight_config_for_all_seeds(
    "capped_inverse_frequency_cap5"
)

PHASE 4 CLASS-WEIGHT CONTROL
Experiment: capped_inverse_frequency_cap5
Scheme:     capped_inverse_frequency
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_capped_inverse_frequency_cap5_seed42_20260702_091025


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Class weights:
anger      count=3878   weight=1.566512
disgust    count=497    weight=5.857051
fear       count=515    weight=5.857051
joy        count=12920  weight=0.470196
sadness    count=2121   weight=2.864183
surprise   count=3553   weight=1.709804
neutral    count=12818  weight=0.473938

Max/min weight ratio: 12.457

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.017678,0.974574,0.631405,0.573671,0.643715
2,0.874620,0.948568,0.665714,0.602443,0.673377
3,0.687542,0.982744,0.663954,0.600225,0.669708


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6046510274218138

Test macro-F1:
0.6025982331025899

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_capped_inverse_frequency_cap5_seed1_20260702_092504


In [ ]:
effective_0999_results = run_one_class_weight_config_for_all_seeds(
    "effective_number_beta0999"
)

PHASE 4 CLASS-WEIGHT CONTROL
Experiment: effective_number_beta0999
Scheme:     effective_number
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_effective_number_beta0999_seed42_20260702_093620


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Class weights:
anger      count=3878   weight=0.967503
disgust    count=497    weight=2.418392
fear       count=515    weight=2.353195
joy        count=12920  weight=0.947524
sadness    count=2121   weight=1.076466
surprise   count=3553   weight=0.975407
neutral    count=12818  weight=0.947525

Max/min weight ratio: 2.552

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.896883,0.851136,0.687926,0.599789,0.688098
2,0.783879,0.838035,0.699802,0.626817,0.699667
3,0.654208,0.856041,0.700022,0.631484,0.700320


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6291745378140023

Test macro-F1:
0.636922686688883

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_effective_number_beta0999_seed1_20260702_095103


In [ ]:
effective_09999_results = run_one_class_weight_config_for_all_seeds(
    "effective_number_beta09999"
)

PHASE 4 CLASS-WEIGHT CONTROL
Experiment: effective_number_beta09999
Scheme:     effective_number
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_effective_number_beta09999_seed42_20260702_095835


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Class weights:
anger      count=3878   weight=1.239505
disgust    count=497    weight=8.217736
fear       count=515    weight=7.937595
joy        count=12920  weight=0.549372
sadness    count=2121   weight=2.084804
surprise   count=3553   weight=1.332415
neutral    count=12818  weight=0.551513

Max/min weight ratio: 14.958

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.010614,0.975262,0.651638,0.576800,0.664295
2,0.873806,0.948620,0.677150,0.608779,0.683287
3,0.678308,0.983396,0.672751,0.603893,0.677808


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6095745047818159

Test macro-F1:
0.6177567879976982

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_effective_number_beta09999_seed1_20260702_101320


In [ ]:
calib_025_results = run_one_class_weight_config_for_all_seeds(
    "calibrated_power025_cap3"
)

PHASE 4 CLASS-WEIGHT CONTROL
Experiment: calibrated_power025_cap3
Scheme:     power_capped
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_calibrated_power025_cap3_seed42_20260702_102043


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Class weights:
anger      count=3878   weight=1.181350
disgust    count=497    weight=1.974428
fear       count=515    weight=1.956945
joy        count=12920  weight=0.874409
sadness    count=2121   weight=1.373710
surprise   count=3553   weight=1.207485
neutral    count=12818  weight=0.876144

Max/min weight ratio: 2.258

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.928834,0.876584,0.690125,0.623497,0.692931
2,0.808572,0.863026,0.696943,0.633829,0.698924
3,0.678367,0.882913,0.690785,0.628753,0.692432


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6318408017780001

Test macro-F1:
0.6350556324084368

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_calibrated_power025_cap3_seed1_20260702_103519


In [ ]:
calib_050_results = run_one_class_weight_config_for_all_seeds(
    "calibrated_power050_cap3"
)

PHASE 4 CLASS-WEIGHT CONTROL
Experiment: calibrated_power050_cap3
Scheme:     power_capped
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_calibrated_power050_cap3_seed42_20260702_104242


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Class weights:
anger      count=3878   weight=1.334040
disgust    count=497    weight=3.460810
fear       count=515    weight=3.460810
joy        count=12920  weight=0.730872
sadness    count=2121   weight=1.803858
surprise   count=3553   weight=1.393719
neutral    count=12818  weight=0.733774

Max/min weight ratio: 4.735

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.981781,0.928409,0.679789,0.609593,0.687022
2,0.852406,0.910239,0.692105,0.629653,0.696044
3,0.694822,0.931759,0.685727,0.622834,0.688809


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6246835551255724

Test macro-F1:
0.6302044188155088

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_calibrated_power050_cap3_seed1_20260702_105731


In [ ]:
calib_075_results = run_one_class_weight_config_for_all_seeds(
    "calibrated_power075_cap3"
)

PHASE 4 CLASS-WEIGHT CONTROL
Experiment: calibrated_power075_cap3
Scheme:     power_capped
Seed:       42
Output:     /content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_calibrated_power075_cap3_seed42_20260702_110452


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Class weights:
anger      count=3878   weight=1.517859
disgust    count=497    weight=3.661712
fear       count=515    weight=3.661712
joy        count=12920  weight=0.615518
sadness    count=2121   weight=2.386612
surprise   count=3553   weight=1.620843
neutral    count=12818  weight=0.619187

Max/min weight ratio: 5.949

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.005639,0.950758,0.658456,0.597489,0.667977
2,0.867057,0.929966,0.678689,0.615879,0.684660
3,0.701684,0.954632,0.674951,0.616299,0.679369


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Validation macro-F1:
0.6158410273726604

Test macro-F1:
0.6188429516802109

Results saved in:
/content/drive/MyDrive/Thesis/models/phase4_class_weight_control_runs/phase4_cw_calibrated_power075_cap3_seed1_20260702_111935


In [ ]:
def collect_class_weight_results(
    output_root=OUTPUT_ROOT,
):
    rows = []

    for result_file in Path(output_root).glob(
        "phase4_cw_*/complete_results.json"
    ):
        with open(
            result_file,
            "r",
            encoding="utf-8",
        ) as file:
            result = json.load(file)

        weight_report = result["weight_report"]

        rows.append({
            "run_name": result["run_name"],
            "experiment_name": result["experiment_name"],
            "seed": result["seed"],
            "max_min_weight_ratio": weight_report["max_min_ratio"],
            "best_validation_macro_f1": result[
                "best_validation_macro_f1_during_training"
            ],
            "validation_macro_f1": result["validation"]["macro_f1"],
            "validation_accuracy": result["validation"]["accuracy"],
            "test_macro_f1": result["test"]["macro_f1"],
            "test_accuracy": result["test"]["accuracy"],
            "path": str(result_file.parent),
        })

    summary = pd.DataFrame(rows)

    if len(summary) == 0:
        raise FileNotFoundError(
            "No class-weight results found."
        )

    summary = summary.sort_values(
        [
            "experiment_name",
            "seed",
        ]
    ).reset_index(drop=True)

    summary.to_csv(
        Path(output_root)
        / "class_weight_controls_by_seed.csv",
        index=False,
    )

    return summary


cw_by_seed = collect_class_weight_results(
    OUTPUT_ROOT
)

cw_by_seed

,run_name,experiment_name,seed,max_min_weight_ratio,best_validation_macro_f1,validation_macro_f1,validation_accuracy,test_macro_f1,test_accuracy,path
0,phase4_cw_calibrated_power025_cap3_seed0_20260...,calibrated_power025_cap3,0,2.258013,0.634778,0.634778,0.698922,0.626953,0.693246,/content/drive/MyDrive/Thesis/models/phase4_cl...
1,phase4_cw_calibrated_power025_cap3_seed1_20260...,calibrated_power025_cap3,1,2.258013,0.631841,0.631841,0.696943,0.635056,0.689325,/content/drive/MyDrive/Thesis/models/phase4_cl...
2,phase4_cw_calibrated_power025_cap3_seed42_2026...,calibrated_power025_cap3,42,2.258013,0.633829,0.633829,0.696943,0.633360,0.693028,/content/drive/MyDrive/Thesis/models/phase4_cl...
3,phase4_cw_calibrated_power050_cap3_seed0_20260...,calibrated_power050_cap3,0,4.735177,0.622030,0.622030,0.683308,0.619869,0.674074,/content/drive/MyDrive/Thesis/models/phase4_cl...
4,phase4_cw_calibrated_power050_cap3_seed1_20260...,calibrated_power050_cap3,1,4.735177,0.624684,0.624684,0.687486,0.630204,0.681699,/content/drive/MyDrive/Thesis/models/phase4_cl...
5,phase4_cw_calibrated_power050_cap3_seed42_2026...,calibrated_power050_cap3,42,4.735177,0.629653,0.629653,0.692105,0.622378,0.677996,/content/drive/MyDrive/Thesis/models/phase4_cl...
6,phase4_cw_calibrated_power075_cap3_seed0_20260...,calibrated_power075_cap3,0,5.948997,0.615721,0.615721,0.674511,0.612917,0.661220,/content/drive/MyDrive/Thesis/models/phase4_cl...
7,phase4_cw_calibrated_power075_cap3_seed1_20260...,calibrated_power075_cap3,1,5.948997,0.615841,0.615841,0.678029,0.618843,0.666885,/content/drive/MyDrive/Thesis/models/phase4_cl...
8,phase4_cw_calibrated_power075_cap3_seed42_2026...,calibrated_power075_cap3,42,5.948997,0.616299,0.616299,0.674951,0.612858,0.660131,/content/drive/MyDrive/Thesis/models/phase4_cl...
9,phase4_cw_capped_inverse_frequency_cap3_seed0_...,capped_inverse_frequency_cap3,0,7.473968,0.610810,0.610810,0.669013,0.606959,0.654466,/content/drive/MyDrive/Thesis/models/phase4_cl...


In [ ]:
cw_mean_std = (
    cw_by_seed
    .groupby("experiment_name")
    .agg(
        seeds=("seed", "count"),
        max_min_weight_ratio_mean=("max_min_weight_ratio", "mean"),
        validation_macro_f1_mean=("validation_macro_f1", "mean"),
        validation_macro_f1_std=("validation_macro_f1", "std"),
        test_macro_f1_mean=("test_macro_f1", "mean"),
        test_macro_f1_std=("test_macro_f1", "std"),
        test_accuracy_mean=("test_accuracy", "mean"),
        test_accuracy_std=("test_accuracy", "std"),
    )
    .reset_index()
    .sort_values(
        "validation_macro_f1_mean",
        ascending=False,
    )
)

cw_mean_std.to_csv(
    OUTPUT_ROOT / "class_weight_controls_mean_std.csv",
    index=False,
)

cw_mean_std

,experiment_name,seeds,max_min_weight_ratio_mean,validation_macro_f1_mean,validation_macro_f1_std,test_macro_f1_mean,test_macro_f1_std,test_accuracy_mean,test_accuracy_std
0,calibrated_power025_cap3,3,2.258013,0.633483,0.001499,0.631790,0.004274,0.691866,0.002204
5,effective_number_beta0999,3,2.552328,0.631024,0.001668,0.635366,0.001562,0.691503,0.001213
1,calibrated_power050_cap3,3,4.735177,0.625456,0.003869,0.624150,0.005391,0.677923,0.003813
7,sqrt_inverse_frequency,3,5.098625,0.624783,0.004447,0.622167,0.003871,0.676471,0.002397
2,calibrated_power075_cap3,3,5.948997,0.615954,0.000305,0.614873,0.003438,0.662745,0.003626
3,capped_inverse_frequency_cap3,3,7.473968,0.609835,0.000974,0.607269,0.000343,0.654248,0.000377
6,effective_number_beta09999,3,14.958423,0.607403,0.003098,0.613089,0.006004,0.666376,0.004159
4,capped_inverse_frequency_cap5,3,12.456614,0.602210,0.002565,0.602140,0.003930,0.651489,0.002193


In [ ]:
calibrated_candidates = cw_mean_std[
    cw_mean_std["experiment_name"].str.startswith(
        "calibrated_"
    )
].copy()

best_calibrated = calibrated_candidates.sort_values(
    "validation_macro_f1_mean",
    ascending=False,
).iloc[0]

best_calibrated

,0
experiment_name,calibrated_power025_cap3
seeds,3
max_min_weight_ratio_mean,2.258013
validation_macro_f1_mean,0.633483
validation_macro_f1_std,0.001499
test_macro_f1_mean,0.63179
test_macro_f1_std,0.004274
test_accuracy_mean,0.691866
test_accuracy_std,0.002204
